In [ ]:
import os, psutil
import time
import json
import pickle
import pandas as pd
import numpy as np
from math import ceil

from functools import partial
from itertools import chain
import joblib
from scipy.sparse import save_npz


from datetime import datetime
from tqdm import tqdm
from dotenv import load_dotenv
from pathlib import Path

In [2]:
import nltk
import networkx as nx
from collections import Counter
from text2graphapi.src.IntegratedSyntacticGraph import ISG

import torch

import optuna
import mlflow
from databricks.sdk import WorkspaceClient

from joblib import Parallel, delayed
import logging

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


2025-12-23 12:29:56,795; - DEBUG; - Import libraries/modules from :PROD


In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

True

Define path variables

In [5]:
representation_type = "integrated_syntactic_graph"
developer_initials = "JP"

In [6]:
current_dir = Path.cwd()
env_path = current_dir.parent.parent / "conf" / "local" / ".env"
results_path = current_dir.parent.parent / "results" / "graph"

train_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-training-large-cleaned.jsonl"
validation_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan20-authorship-verification-validation-large-cleaned.jsonl"
test_data_full_cleaned_path = current_dir.parent.parent / "data" /  "01_processed" / "pan21-authorship-verification-test-cleaned.jsonl"

vocabulary_index_path = current_dir.parent.parent / "data" / "02_models" / "graph" / "vocab_index.pkl"

train_data_isg1_path = current_dir.parent.parent / "data" / "01_processed" / "training-vectors-isg1.dat"
train_data_isg2_path = current_dir.parent.parent / "data" / "01_processed" / "training-vectors-isg2.dat"

val_data_isg1_path = current_dir.parent.parent / "data" / "01_processed" / "val-vectors-isg1.dat"
val_data_isg2_path = current_dir.parent.parent / "data" / "01_processed" / "val-vectors-isg2.dat"

test_data_isg1_path = current_dir.parent.parent / "data" / "01_processed" / "test-vectors-isg1.dat"
test_data_isg2_path = current_dir.parent.parent / "data" / "01_processed" / "test-vectors-isg2.dat"

Connect to databricks for logging results

In [7]:
load_dotenv(env_path)

w = WorkspaceClient()   
print("Connected to:", w.config.host)

mlflow.set_tracking_uri("databricks")
mlflow.autolog()

2025/12/23 12:29:58 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.4.0 <= scikit-learn, but the installed version is 1.3.2. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2025/12/23 12:29:58 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2025/12/23 12:29:58 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.1 <= statsmodels, but the installed version is 0.14.0. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.


Connected to: https://dbc-1ea3ad0e-f504.cloud.databricks.com


2025/12/23 12:29:58 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.


What are GPU are the experiments run on

In [8]:
!nvidia-smi

Tue Dec 23 12:29:58 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     Off |   00000000:A3:00.0 Off |                    0 |
|  0%   36C    P8             30W /  300W |       3MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
running_on_gpu = torch.cuda.is_available()

In [10]:
if running_on_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_vram_gb = round(gpu_props.total_memory / (1024**3), 2)
else:
    gpu_name = "CPU"
    gpu_props = "N/A"
    gpu_vram_gb = 0

Empty the GPU from previous experiments

In [11]:
import gc
import torch
torch.cuda.empty_cache()
gc.collect()

89

In [12]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["NUMEXPR_MAX_THREADS"] = "8"
os.environ["NUMEXPR_NUM_THREADS"] = "8"

Classification threshold constant specification

In [13]:
classification_thresholds = [x/1000 for x in range(200, 999)]

# Load dataset

#### Load training data

In [14]:
train_data_file_size = os.path.getsize(train_data_full_cleaned_path)
train_data = []

with open(train_data_full_cleaned_path, 'r') as f:
    with tqdm(total=train_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            train_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(train_data)} items.")

Loading data: 100%|██████████| 11.3G/11.3G [00:24<00:00, 453MB/s]


Successfully loaded 273301 items.


In [15]:
train_data_df = pd.DataFrame(train_data)

Prepare dataset for cosine embedding loss

In [16]:
train_data_df.head(10)

,id,pair,same
0,e05b9c0b-88a1-5608-b7e8-ab1fc6b78dc1,"[Well, ever since you and Kurt broke up youve ...",False
1,12f73a20-cdf3-58df-b5bb-9392eec9b486,"[The thing is, Ryouga has no reason to run aft...",False
2,d82c6764-451b-544c-8711-c139e9349c56,"[Ehhhh nah, its silly' Its my job to listen to...",True
3,876b8380-9260-5427-93e8-dc31155c3edd,"[Glaring at the arrogant spark, Always asks va...",False
4,357e8471-35b9-50b4-9ac0-286ac0e8b101,[Runa limped across the small space to an open...,False
5,6b39fe22-409f-5329-9fe1-ddf4a70bedd1,"[And thats retired Commander, if you please St...",False
6,76ed2017-0c9f-580f-83b1-5a041a8169ec,[Meet you downstairs in twenty minutes I say w...,True
7,fd8deb2e-06de-5c92-9a4c-927891c53657,"[Since Ive seen so many others do so, Im going...",False
8,3ce5e811-a57c-5fbf-9f9a-2ee636e50be6,[party Eishi exclaimed Omi shook his head and ...,False
9,25a17cd2-6b01-5fba-99ef-e631e56e181d,[After a few moments she found that Red was ri...,False


#### Load validation data

In [17]:
val_data_file_size = os.path.getsize(validation_data_full_cleaned_path)
val_data = []

with open(validation_data_full_cleaned_path, 'r') as f:
    with tqdm(total=val_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            val_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(val_data)} items.")

Loading data: 100%|██████████| 103M/103M [00:00<00:00, 474MB/s] 


Successfully loaded 2500 items.


In [18]:
val_data_df = pd.DataFrame(val_data)

#### Load testing data

In [19]:
test_data_file_size = os.path.getsize(test_data_full_cleaned_path)
test_data = []

with open(test_data_full_cleaned_path, 'r') as f:
    with tqdm(total=test_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data)} items.")

Loading data: 100%|██████████| 826M/826M [00:02<00:00, 305MB/s] 


Successfully loaded 19999 items.


In [20]:
test_data_df = pd.DataFrame(test_data)

# Functions to build graphs and extract features

In [21]:
def texts_to_isg_graphs(texts, n_jobs=-1):
    def process(id, text):
        
        logging.disable(logging.INFO)
        logging.getLogger('text2graphapi').setLevel(logging.WARNING)
        logging.getLogger('text2graphapi.models').setLevel(logging.WARNING)
        
        isg = ISG(
            graph_type="DiGraph",
            language="en",
            apply_prep=True,
            output_format="networkx"
        )
        corpus = [{"id": id, "doc": text}]
        graph_object = isg.transform(corpus)[0]["graph"]
        return graph_object

    graphs = Parallel(n_jobs=n_jobs)(
        delayed(process)(id, text)
        for id, text in tqdm(
            enumerate(texts),
            total=len(texts),
            desc="Processing ISGs, print_"
        )
    )

    return graphs

Parse ISG's nodes POS and lemma function

In [22]:
def parse_graph_node(graph_node):
    node_str = str(graph_node)
    if "_" in node_str:
        lemma, pos = node_str.rsplit("_", 1)
        return lemma.lower(), pos

Parse dependency

In [23]:
def parse_graph_dependency(data):
    dependency = data.get("gramm_relation")
    parsed_dependency = dependency.split("_", 1)[0]
    return parsed_dependency

Extract multi-level features from graph

In [24]:
def extract_features_from_isg(graph):
    features = Counter()

    for node in graph.nodes:
        lemma, pos = parse_graph_node(node)
        features[f"LEX::{lemma}"] += 1
        
        if pos:
            features[f"POS::{pos}"] += 1

    for _, _, data in graph.edges(data=True):
        dependency = parse_graph_dependency(data)
        if dependency:
            features[f"DEP::{dependency}"] += 1

    return features

Build vectors based on vocabulary

In [25]:
def build_vector(features, index):
    vector = np.zeros(len(index), dtype=np.float32)
    for feature, value in features.items():
        if feature in index:
            vector[index[feature]] = value
    return vector

Print process RAM usage

In [26]:
def print_ram_usage():
    print(f"Process RAM usage: {process.memory_info().rss / 1e9:.2f} GB")

Convert texts to text2graphapi integrated syntactic graphs

In [27]:
def convert_texts_to_vectors(input_df, index, n_jobs=1, batch_size=3000):
    vectors1 = []
    vectors2 = []
    
    texts1 = input_df["pair"].apply(lambda x: x[0])
    total_batches = ceil(len(texts1) / batch_size)

    for batch_index in tqdm(
    range(total_batches),
    desc="#1 in pair - Processing text batches"
    ):
        start = batch_index * batch_size
        end = start + batch_size
        batch = texts1[start:end]
    
        graphs = texts_to_isg_graphs(batch, n_jobs)
    
        for graph in graphs:
            features = extract_features_from_isg(graph)
            vectors1.append(build_vector(features, index))
    
        del graphs
        gc.collect()
        print_ram_usage()
    
    
    texts2 = input_df["pair"].apply(lambda x: x[1])
    
    for batch_index in tqdm(
        range(total_batches),
        desc="#2 in pair - processing text batches"
    ):
        start = batch_index * batch_size
        end = start + batch_size
        batch = texts2[start:end]
    
        graphs = texts_to_isg_graphs(batch, n_jobs)
    
        for graph in graphs:
            features = extract_features_from_isg(graph)
            vectors2.append(build_vector(features, index))
    
        del graphs
        gc.collect()
        print_ram_usage()
    
    return vectors1, vectors2

# Build vocabulary index

Build index from the training texts

In [28]:
def build_index(train_df, n_jobs=1, batch_size=3000):
    index = {}
    next_index_value = 0

    #all training texts
    texts = (
        train_df["pair"].apply(lambda x: x[0]).tolist() +
        train_df["pair"].apply(lambda x: x[1]).tolist()
    )
    
    # total number of batches needed to build the index
    total_batches = ceil(len(texts) / batch_size)

    for batch_index in tqdm(
        range(total_batches),
        desc="Processing batches of training texts"
    ):
        #the first index of a given batch
        start = batch_index * batch_size

        #the last index of a given batch
        end = start + batch_size
        
        #all the texts between the first and last index
        batch = texts[start:end]

        graphs = texts_to_isg_graphs(batch, n_jobs)

        for graph in graphs:
            features =  extract_features_from_isg(graph)
            for feature in features:
                if feature not in index:
                    index[feature] = next_index_value
                    next_index_value += 1

        del graphs
        gc.collect()
        print(len(index))
        print_ram_usage()

    return index

Build index for building vectors first - separately to conserve RAM

In [ ]:
process = psutil.Process(os.getpid())
index = build_index(train_data_df, n_jobs=32, batch_size=3000)

Processing ISGs, print_:   0%|          | 0/3000 [00:00<?, ?it/s][nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading packag

2025-12-23 12:30:31,627; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:31,660; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:31,768; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:31,801; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:31,803; - DEBUG; - Import libraries/modules from :PROD


[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


2025-12-23 12:30:31,885; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:31,898; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:31,922; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:31,967; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:31,979; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:32,011; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:32,024; - DEBUG; - Import libraries/modules from :PROD
2025-12-23 12:30:32,055; - DEBUG; - Import libraries/modules from :PROD


[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


...

2025-12-23 13:36:07,591; - DEBUG; - Import libraries/modules from :PROD


[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


2025-12-23 13:36:11,685; - DEBUG; - Import libraries/modules from :PROD



Processing ISGs, print_:  25%|██▍       | 736/3000 [00:47<03:14, 11.62it/s][nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!

Processing ISGs, print_:  26%|██▌       | 768/3000 [00:48<02:31, 14.71it/s]

2025-12-23 13:36:13,643; - DEBUG; - Import libraries/modules from :PROD


In [ ]:
len(index)

In [ ]:
with open(vocabulary_index_path, "wb") as f:
    pickle.dump(index, f)

# Build the graphs for texts in pair and then vectors out of them and the vocabulary index

In [ ]:
index = None
with open(vocabulary_index_path, "rb") as f:
    index = pickle.load(f)

In [ ]:
train_vectors1, train_vectors2 = convert_texts_to_vectors(train_data_df, index, n_jobs=8, batch_size=3000)

In [ ]:
train_vectors1 = np.asarray(train_vectors1, dtype="float32")
train_vectors2 = np.asarray(train_vectors2, dtype="float32")

save_npz("X_left.npz",  X_left)

In [ ]:
del train_embeddings,  train_vectors1,  train_vectors2

# Create vectors for testing and validation data and save them

Convert validation data to vectors

In [ ]:
val_vectors1, val_vectors2 = convert_texts_to_vectors(val_data_df, index, n_jobs=8, batch_size=3000)

In [ ]:
del val_embeddings,  val_vectors1,  val_vectors2

Convert test data to vectors

In [ ]:
test_vectors1, test_vectors2 = convert_texts_to_vectors(test_data_df, index, n_jobs=8, batch_size=3000)

In [ ]:
del test_embeddings,  test_vectors1,  test_vectors2